In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os

cwd = Path.cwd()
env_path = cwd / ".env" if (cwd / ".env").exists() else cwd.parent / ".env"

print("env_path =", env_path)
print("exists =", env_path.exists())
print("loaded =", load_dotenv(env_path, override=True))
print("api_key =", repr(os.getenv("OPENAI_API_KEY")[:10] if os.getenv("OPENAI_API_KEY") else None))

env_path = c:\Users\USER\Desktop\complypilot-jb\.env
exists = True
loaded = True
api_key = 'sk-proj-61'


In [2]:
# [중요] 폴더 전체를 읽어오기 위한 라이브러리 추가
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

import os
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

pdf_dir = PROJECT_ROOT / "data" / "vectordb"
db_path = PROJECT_ROOT / "data" / "chromadb"
db_path.mkdir(parents=True, exist_ok=True)

loader = DirectoryLoader(
    path=str(pdf_dir),
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
)

print("폴더 내 모든 PDF 파일 로드 시작...")
documents_pdf = loader.load()
print(f"총 {len(documents_pdf)} 개의 전체 페이지가 로드됨")

# 2. 텍스트 분할 (기존 코드와 동일)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 900,       # 조항이 통째로 들어가도록 크기를 900으로 확대
    chunk_overlap = 150,
    separators = ["\n\n", "\n", " ", ""]
)
split_docs = text_splitter.split_documents(documents_pdf)
print(f"분할된 총 청크 개수: {len(split_docs)}")

# 3. 임베딩 및 벡터스토어 생성 (기존 코드와 동일)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=split_docs, 
    embedding=embedding_model,
    persist_directory=str(db_path)
)

print("모든 문서가 Vector DB에 성공적으로 저장되었습니다!")

C:\Users\USER\AppData\Local\Temp\ipykernel_11852\1003840607.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


폴더 내 모든 PDF 파일 로드 시작...
총 186 개의 전체 페이지가 로드됨
분할된 총 청크 개수: 343
모든 문서가 Vector DB에 성공적으로 저장되었습니다!


In [1]:
# 고도화 시킨 vector db

# ============================================================
# Vector DB 재생성
# 목적:
# - data/vectordb 안의 PDF 전체를 Chroma DB로 저장
# - 문서명, 페이지, chunk_id metadata를 강화
# - 기존 DB를 삭제하고 깨끗하게 재생성
# ============================================================

from __future__ import annotations

import os
import re
import shutil
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PDF_DIR = PROJECT_ROOT / "data" / "vectordb"
CHROMA_DB_DIR = PROJECT_ROOT / "data" / "chromadb"

COLLECTION_NAME = "complypilot_regulations"


def normalize_vector_text(text: str) -> str:
    """
    Vector DB 저장 전 텍스트를 정리한다.

    Args:
        text: 원본 텍스트

    Returns:
        str: 정규화된 텍스트
    """
    if not text:
        return ""

    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def rebuild_chroma_dir(chroma_dir: Path) -> None:
    """
    기존 Chroma DB 폴더를 삭제하고 새로 생성한다.

    Args:
        chroma_dir: Chroma DB 저장 경로

    Returns:
        None
    """
    if chroma_dir.exists():
        shutil.rmtree(chroma_dir)

    chroma_dir.mkdir(parents=True, exist_ok=True)


print("PROJECT_ROOT:", PROJECT_ROOT)
print("PDF_DIR:", PDF_DIR)
print("CHROMA_DB_DIR:", CHROMA_DB_DIR)

if not PDF_DIR.exists():
    raise FileNotFoundError(f"PDF_DIR이 존재하지 않습니다: {PDF_DIR}")

pdf_files = sorted(PDF_DIR.rglob("*.pdf"))

print(f"PDF 파일 개수: {len(pdf_files)}")
for path in pdf_files:
    print("-", path.relative_to(PROJECT_ROOT))

if not pdf_files:
    raise ValueError("data/vectordb 안에 PDF 파일이 없습니다.")


# 1. 기존 Chroma DB 삭제 후 재생성
rebuild_chroma_dir(CHROMA_DB_DIR)


# 2. PDF 전체 로딩
loader = DirectoryLoader(
    path=str(PDF_DIR),
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True,
)

print("\nPDF 로드 시작...")
documents_pdf = loader.load()
print(f"총 {len(documents_pdf)}개 페이지 로드 완료")


# 3. 페이지 텍스트 정리 및 metadata 보강
cleaned_documents = []

for doc in documents_pdf:
    text = normalize_vector_text(doc.page_content)

    if len(text) < 30:
        continue

    source_path = doc.metadata.get("source", "")
    doc_title = Path(source_path).name if source_path else ""

    # PyMuPDFLoader의 page는 보통 0-index이므로 표시용으로 +1
    raw_page = doc.metadata.get("page", None)
    display_page = raw_page + 1 if isinstance(raw_page, int) else raw_page

    doc.page_content = text
    doc.metadata.update(
        {
            "source": source_path,
            "doc_title": doc_title,
            "page": display_page,
            "raw_page": raw_page,
        }
    )

    cleaned_documents.append(doc)

print(f"정리 후 문서 페이지 수: {len(cleaned_documents)}")


# 4. 텍스트 분할
# 금융광고 규정 문서는 조항 단위가 길 수 있으므로 800~1000 사이가 적당
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=850,
    chunk_overlap=150,
    separators=["\n\n", "\n", "다.", ". ", " ", ""],
)

split_docs = text_splitter.split_documents(cleaned_documents)

# 5. chunk metadata 보강 + 검색 품질 개선용 title prefix 추가
final_docs = []

for idx, doc in enumerate(split_docs):
    source_path = doc.metadata.get("source", "")
    doc_title = doc.metadata.get("doc_title") or Path(source_path).name
    page = doc.metadata.get("page")

    cleaned_text = normalize_vector_text(doc.page_content)

    if len(cleaned_text) < 30:
        continue

    # 문서명/페이지를 chunk 앞에 붙이면 general query 검색 품질이 조금 좋아질 수 있음
    enriched_text = f"[문서명: {doc_title} / 페이지: {page}]\n{cleaned_text}"

    doc.page_content = enriched_text
    doc.metadata.update(
        {
            "chunk_id": idx,
            "doc_title": doc_title,
            "page": page,
            "text_length": len(cleaned_text),
        }
    )

    final_docs.append(doc)

print(f"분할된 최종 청크 개수: {len(final_docs)}")


# 6. 임베딩 및 Chroma DB 생성
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=final_docs,
    embedding=embedding_model,
    persist_directory=str(CHROMA_DB_DIR),
    collection_name=COLLECTION_NAME,
)

print("\nVector DB 생성 완료")
print("저장 위치:", CHROMA_DB_DIR)
print("collection_name:", COLLECTION_NAME)

C:\Users\USER\AppData\Local\Temp\ipykernel_7992\3671420585.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


PROJECT_ROOT: c:\Users\USER\Desktop\complypilot-jb
PDF_DIR: c:\Users\USER\Desktop\complypilot-jb\data\vectordb
CHROMA_DB_DIR: c:\Users\USER\Desktop\complypilot-jb\data\chromadb
PDF 파일 개수: 5
- data\vectordb\금소법 안내 자료_게시용_.pdf
- data\vectordb\금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf
- data\vectordb\금융소비자 보호에 관한 법률(법률)(제21065호)(20260102).pdf
- data\vectordb\별첨자료_금융광고규제가이드라인.pdf
- data\vectordb\표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121).pdf

PDF 로드 시작...


100%|██████████| 5/5 [00:00<00:00,  5.30it/s]


총 186개 페이지 로드 완료
정리 후 문서 페이지 수: 186
분할된 최종 청크 개수: 353

Vector DB 생성 완료
저장 위치: c:\Users\USER\Desktop\complypilot-jb\data\chromadb
collection_name: complypilot_regulations


In [1]:
# ============================================================
# RAG 검색 품질 테스트 + 근거 기반 QA 테스트
# 목적:
# - Chroma Vector DB가 실제로 어떤 근거를 찾는지 확인
# - 검색 점수와 source document를 함께 출력
# - LLM 답변은 근거 문서 기반으로만 생성
# ============================================================

from pathlib import Path

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CHROMA_DB_DIR = PROJECT_ROOT / "data" / "chromadb"
COLLECTION_NAME = "complypilot_regulations"


# ============================================================
# 1. Vector DB 로드
# ============================================================

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    persist_directory=str(CHROMA_DB_DIR),
    embedding_function=embedding_model,
    collection_name=COLLECTION_NAME,
)

print("✅ Vector DB 로드 완료")
print("CHROMA_DB_DIR:", CHROMA_DB_DIR)
print("COLLECTION_NAME:", COLLECTION_NAME)


# ============================================================
# 2. 검색 점수 먼저 확인
# ============================================================

query = "금융광고를 할 때 필수적으로 포함해야 하는 문구나 정보에는 어떤 것들이 있어?"

print("\n[Vector DB 검색 점수 확인]")
print("=" * 80)

docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
    query,
    k=7,
)

for i, (doc, score) in enumerate(docs_with_scores, start=1):
    source = doc.metadata.get("source", "")
    doc_title = doc.metadata.get("doc_title") or Path(source).name
    page = doc.metadata.get("page", None)

    print(f"\n📄 [검색 결과 {i}]")
    print("score:", round(float(score), 3))
    print("doc_title:", doc_title)
    print("page:", page)
    print("source:", source)
    print("context:")
    print(doc.page_content[:500])
    print("-" * 80)


# ============================================================
# 3. Retriever 생성
# - similarity보다 MMR이 중복 문서 감소에 유리
# ============================================================

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    },
)


# ============================================================
# 4. LLM 설정
# - 비용/속도 우선이면 gpt-4o-mini
# - 품질 우선이면 gpt-4o 또는 사용 중인 모델명으로 변경
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


# ============================================================
# 5. RAG Prompt
# - 근거 문서에 없는 내용은 단정하지 않게 제한
# ============================================================

qa_prompt = PromptTemplate.from_template(
    """
너는 금융광고 준법 검토를 돕는 AI야.
아래 [근거 문서]에 포함된 내용만 바탕으로 답변해.

규칙:
- 근거 문서에 없는 내용은 추측하지 말 것
- 확실하지 않으면 "근거 문서만으로는 확인이 어렵습니다"라고 말할 것
- 답변은 한국어로 작성할 것
- 가능하면 항목별로 정리할 것
- 마지막에 어떤 근거에서 판단했는지 간단히 요약할 것

[근거 문서]
{context}

[질문]
{question}

[답변]
"""
)


# ============================================================
# 6. Memory 설정
# ============================================================

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)


# ============================================================
# 7. ConversationalRetrievalChain 구성
# ============================================================

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={
        "prompt": qa_prompt,
    },
)


# ============================================================
# 8. RAG 질의 실행
# ============================================================

response = qa_chain.invoke(
    {
        "question": query,
    }
)


# ============================================================
# 9. 최종 답변 출력
# ============================================================

print("\n\n[최종 답변]")
print("=" * 80)
print(response["answer"])


# ============================================================
# 10. LLM이 참고한 source documents 출력
# ============================================================

print("\n\n[벡터 DB가 찾아온 실제 PDF 근거 본문]")
print("=" * 80)

for i, doc in enumerate(response.get("source_documents", []), start=1):
    source = doc.metadata.get("source", "")
    doc_title = doc.metadata.get("doc_title") or Path(source).name
    page = doc.metadata.get("page", None)

    print(f"\n📄 [근거 문서 {i}]")
    print("doc_title:", doc_title)
    print("page:", page)
    print("source:", source)
    print("context:")
    print(doc.page_content[:600])
    print("-" * 80)

✅ Vector DB 로드 완료
CHROMA_DB_DIR: c:\Users\USER\Desktop\complypilot-jb\data\chromadb
COLLECTION_NAME: complypilot_regulations

[Vector DB 검색 점수 확인]

📄 [검색 결과 1]
score: 0.428
doc_title: 별첨자료_금융광고규제가이드라인.pdf
page: 14
source: c:\Users\USER\Desktop\complypilot-jb\data\vectordb\별첨자료_금융광고규제가이드라인.pdf
context:
[문서명: 별첨자료_금융광고규제가이드라인.pdf / 페이지: 14]
- 12 -
Ⅳ. 광고의 내용 및 방법
관련 법령 주요내용
광고 시 금소법 뿐만 아니라 표시광고법, 방송법, 대부업법 등 
다른 법령에 위배되는 사항이 있는지도 꼼꼼히 확인해야* 함
 * 금소법 제6조(다른 법률과의 관계) 금융소비자 보호에 관하여 다른 법률에서 
특별히 정한 경우를 제외하고는 이 법에서 정하는 바에 따른다
 ㅇ 특히 유튜브, 블로그 등 온라인 매체를 통한 광고 시 뒷광고*(hidden 
ad) 이슈가 발생하지 않도록 최근 공정위에서 개정한 「추천·
보증 등에 관한 표시·광고 심사지침」을 준수해야 할 것임
 * 유명인이 광고를 하면서 광고주와의 경제적 이해관계를 표시하지 않는 경우 등
금소법령상 광고 내용에 포함시키도록 열거된 사항은 광고의 
목적, 광고매체의 특성 등을 감안하여 규제취지를 형해화하지 
않는 범위 내에서 탄력적으로 운영할 수 있음 (☞ 참고1)
 ㅇ 예컨대 온라인 배너·팝업광고는 
--------------------------------------------------------------------------------

📄 [검색 결과 2]
score: 0.423
doc_title: 별첨자료_금융광고규제가이드라인.pdf
page: 5
source: c:\Users\USER\Desktop\complypilot-jb\data

C:\Users\USER\AppData\Local\Temp\ipykernel_25256\791802971.py:129: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(




[최종 답변]
금융광고를 할 때 필수적으로 포함해야 하는 문구나 정보는 다음과 같습니다:

1. **이자율의 범위 및 산출기준**
   - 대출상품 광고 시 소비자가 오인하지 않도록 이자율의 범위 및 산출기준을 명확히 기재해야 함.
   - '변동금리', '금리변동 가능'과 같은 표현은 적합하지 않음.

2. **갖춰야 할 신용 수준**
   - 대출성 상품 광고 시 소비자가 대출 가능 여부를 판단할 수 있는 기준을 제시해야 함.
   - '내부 심사에 따라 대출 가능여부가 달라진다'는 표기는 부적합함.

3. **금소법령상 광고에 포함해야 할 사항**
   - 금소법 제22조제3항에 규정된 사항은 광고에서 제외할 수 없음.
   - 광고의 목적과 매체의 특성을 고려하여 규제 취지를 형해화하지 않는 범위 내에서 탄력적으로 운영 가능.

4. **광고의 방법**
   - 광고 시 글자의 색깔, 크기 또는 음성의 속도, 크기 등을 소비자가 받을 수 있는 혜택과 불이익을 균형 있게 전달할 수 있도록 구성해야 함.

이 외에도 광고 시 다른 법령(표시광고법, 방송법 등)과의 위배 여부를 확인해야 하며, 온라인 매체를 통한 광고 시 뒷광고 이슈를 피하기 위해 공정위의 심사지침을 준수해야 함.

**요약**: 금융광고 시 필수적으로 포함해야 하는 정보는 이자율의 범위 및 산출기준, 신용 수준, 금소법령상 규정된 사항, 광고 방법의 균형성 등입니다. (근거: 별첨자료_금융광고규제가이드라인.pdf, 페이지 14, 18)


[벡터 DB가 찾아온 실제 PDF 근거 본문]

📄 [근거 문서 1]
doc_title: 별첨자료_금융광고규제가이드라인.pdf
page: 14
source: c:\Users\USER\Desktop\complypilot-jb\data\vectordb\별첨자료_금융광고규제가이드라인.pdf
context:
[문서명: 별첨자료_금융광고규제가이드라인.pdf / 페이지: 14]
- 12 -
Ⅳ. 광고의 내용 및 방법
관련 법령 주요내용
광고 시 금소법 뿐만 아

In [3]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

# (1) 리트리버(Retriever) 생성
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 7})

# (2) GPT-4.1-mini 모델 설정
llm = ChatOpenAI(model_name="gpt-4o")

# (3) 메모리 추가 (대화 문맥 유지)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")

# (4) RAG 기반 ConversationalRetrievalChain 구성
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True  # 검색된 문서 출력 옵션
)

C:\Users\USER\AppData\Local\Temp\ipykernel_11852\2802484103.py:12: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")


In [4]:
query = "금융광고를 할 때 필수적으로 포함해야 하는 문구나 정보에는 어떤 것들이 있어?"
response = qa_chain({"question": query})

# (1) 챗봇의 최종 답변 출력
print("[최종 답변]")
print(response["answer"])
print("\n" + "="*50 + "\n")

# (2) 💡 챗봇이 답변을 만들기 위해 벡터 DB에서 찾아온 근거 문서들 출력
print("[벡터 DB가 찾아온 실제 PDF 근거 본문]")
for i, doc in enumerate(response.get("source_documents", [])):
    print(f"\n📄 [근거 문서 {i+1}]")
    # 어떤 PDF 파일의 몇 번째 페이지인지 출처 확인
    print(f"출처 파일: {doc.metadata.get('source')} (Page: {doc.metadata.get('page', 0) + 1})")
    print(f"매칭된 실제 문맥(Context):\n{doc.page_content[:400]}...") # 앞 400자만 출력
    print("-" * 30)

C:\Users\USER\AppData\Local\Temp\ipykernel_11852\1711612605.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = qa_chain({"question": query})


[최종 답변]
금융광고를 할 때 필수적으로 포함해야 하는 문구나 정보는 다음과 같습니다:

1. 계약 체결 전 설명서 및 약관을 읽어볼 것을 권유하는 내용
2. 금융상품판매업자 등의 명칭, 금융상품의 내용
3. 금융상품 유형별 필수 포함사항:
   - 보장성, 투자성, 예금성, 대출성 상품 각각에 따라 필요한 세부 정보
4. 법령 및 내부통제기준에 따른 광고 관련 절차의 준수 여부

이 외에도 특정 금융소비자 보호를 위해 대통령령으로 정하는 사항들도 포함되어야 할 수 있습니다.


[벡터 DB가 찾아온 실제 PDF 근거 본문]

📄 [근거 문서 1]
출처 파일: c:\Users\USER\Desktop\complypilot-jb\data\vectordb\별첨자료_금융광고규제가이드라인.pdf (Page: 14)
매칭된 실제 문맥(Context):
- 12 -
Ⅳ. 광고의 내용 및 방법
관련 법령 주요내용
광고 시 금소법 뿐만 아니라 표시광고법, 방송법, 대부업법 등 
다른 법령에 위배되는 사항이 있는지도 꼼꼼히 확인해야* 함
    * 금소법 제6조(다른 법률과의 관계) 금융소비자 보호에 관하여 다른 법률에서 
특별히 정한 경우를 제외하고는 이 법에서 정하는 바에 따른다
 ㅇ 특히 유튜브, 블로그 등 온라인 매체를 통한 광고 시 뒷광고*(hidden 
ad) 이슈가 발생하지 않도록 최근 공정위에서 개정한 「추천·
보증 등에 관한 표시·광고 심사지침」을 준수해야 할 것임
    * 유명인이 광고를 하면서 광고주와의 경제적 이해관계를 표시하지 않는 경우 등
금소법령상 광고 내용에 포함시키도록 열거된 사항은 광고의 
목적, 광고매체의 특성 등을 감안하...
------------------------------

📄 [근거 문서 2]
출처 파일: c:\Users\USER\Desktop\complypilot-jb\data\vectordb\별첨자료_금융광고규제가이드라인.pdf (Page: 5)
매칭된 실제 문맥(Context):
- 3 -
관련 주요 질의․답변
1. 협

In [5]:
query = query = "대출 광고에서 '최저금리', '누구나 승인', '수수료 무료' 같은 표현은 왜 주의해야 해?"
response = qa_chain({"question": query})

# (1) 챗봇의 최종 답변 출력
print("[최종 답변]")
print(response["answer"])
print("\n" + "="*50 + "\n")

# (2) 💡 챗봇이 답변을 만들기 위해 벡터 DB에서 찾아온 근거 문서들 출력
print("[벡터 DB가 찾아온 실제 PDF 근거 본문]")
for i, doc in enumerate(response.get("source_documents", [])):
    print(f"\n📄 [근거 문서 {i+1}]")
    # 어떤 PDF 파일의 몇 번째 페이지인지 출처 확인
    print(f"출처 파일: {doc.metadata.get('source')} (Page: {doc.metadata.get('page', 0) + 1})")
    print(f"매칭된 실제 문맥(Context):\n{doc.page_content[:400]}...") # 앞 400자만 출력
    print("-" * 30)

[최종 답변]
대출 광고에서 '최저금리', '누구나 승인', '수수료 무료'와 같은 표현은 소비자가 이를 오해할 수 있는 가능성이 있기 때문에 주의해야 합니다. 

1. **'최저금리'**: 이러한 표현은 소비자가 최저금리가 자주 제공되는 조건이라고 오인하게 만들 수 있으며, 금소법에 따라 이자율의 범위 및 산출기준을 명확히 제시하여 소비자가 오해하지 않도록 해야 합니다.

2. **'누구나 승인'**: 이 표현은 모든 신청자가 대출 승인을 받을 수 있다고 오인하게 하며, 이는 잘못된 정보를 제공하는 것이 되기 때문에 금소법령상 금지되고 있습니다. 대출 승인은 보통 특정 신용 수준 등의 조건을 충족해야 승인될 수 있기 때문입니다.

3. **'수수료 무료'**: 광고에서 이러한 표현을 사용할 때는 실제로 소비자에게 적용되는 모든 요금과 수수료를 명확히 알리지 않을 경우 오해를 일으킬 수 있습니다. 이는 상담 시 명확한 정보를 제공해야 하는 이유입니다.

따라서 이러한 표현을 사용할 때는 오해를 방지하기 위해 명확하고 정확한 정보를 제공해야 합니다.


[벡터 DB가 찾아온 실제 PDF 근거 본문]

📄 [근거 문서 1]
출처 파일: c:\Users\USER\Desktop\complypilot-jb\data\vectordb\별첨자료_금융광고규제가이드라인.pdf (Page: 5)
매칭된 실제 문맥(Context):
- 3 -
관련 주요 질의․답변
1. 협회의 금융상품 정보 비교공시 서비스가 금소법상 광고에 
해당하는지?
□협회의 금융상품 정보 비교공시 서비스는 금소법에 따라 공익 
목적으로 제공된다는 점에서 광고로 보기 어려움
2. 금융정보 제공 방송도 금소법상 광고에 해당하는지?
□특정 금융상품판매업자의 금융상품에 관한 정보를 직·간접적
으로 제공하는 방송은 “금융상품 광고”로 볼 수 있음
 ㅇ 다만, 판매의도 없이 소비자가 금융상품판매업자나 금융상품을 
쉽게 유추할 수 없도록 조치(예: “A社”로 익명처리)하여 금융
정보를 제공하는 경우에는 광고로 보기 어려움